In [2]:
!pip install -q --upgrade trl transformers peft bitsandbytes accelerate wandb huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 15.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 85.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 53.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 63.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 68.7 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of t

In [3]:
# ── Cell 2: Auth and config ──────────────────────────────────────────────────
import os
import torch
import wandb
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

os.environ["HF_TOKEN"] = "REMOVED_SECRET"
os.environ["WANDB_API_KEY"] = "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"

login(token=os.environ["HF_TOKEN"])
wandb.login(key=os.environ["WANDB_API_KEY"])

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tesla369 (tesla369-pdepth) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


GPU: Tesla T4
Memory: 15.6 GB


In [4]:
# ── Cell 3: Load dataset ─────────────────────────────────────────────────────
dataset = load_dataset("TeslaInch/SCD-Instruction-Tuning")

print(dataset)
print(f"\nTrain: {len(dataset['train'])}")
print(f"Validation: {len(dataset['validation'])}")
print(f"\nSample:")
print(dataset['train'][0])

README.md:   0%|          | 0.00/531 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/122k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/18.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/353 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/39 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'source', 'category'],
        num_rows: 353
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output', 'source', 'category'],
        num_rows: 39
    })
})

Train: 353
Validation: 39

Sample:
{'instruction': 'You are a medical AI assistant specialised in sickle cell disease. Answer the following clinical question accurately and completely.', 'input': 'All of the following statements about sickle cell disease are true, EXCEPT:', 'output': "Sticky patch' is generated as a result of replacement of a non polar residue with a polar residue", 'source': 'layer2_mcq_filtered', 'category': 'multiple_choice'}


In [5]:
# ── Cell 4: Format for Phi-3.5 ──────────────────────────────────────────────
def format_example(example):
    text = f"<|user|>\n{example['instruction']}\n{example['input']}<|end|>\n<|assistant|>\n{example['output']}<|end|>"
    return {"text": text}

train_formatted = dataset['train'].map(format_example)
val_formatted = dataset['validation'].map(format_example)

print(f"Train formatted: {len(train_formatted)}")
print(f"Val formatted: {len(val_formatted)}")
print(f"\nSample formatted:")
print(train_formatted[0]['text'][:500])

Map:   0%|          | 0/353 [00:00<?, ? examples/s]

Map:   0%|          | 0/39 [00:00<?, ? examples/s]

Train formatted: 353
Val formatted: 39

Sample formatted:
<|user|>
You are a medical AI assistant specialised in sickle cell disease. Answer the following clinical question accurately and completely.
All of the following statements about sickle cell disease are true, EXCEPT:<|end|>
<|assistant|>
Sticky patch' is generated as a result of replacement of a non polar residue with a polar residue<|end|>


In [6]:
# ── Cell 5: Load model ───────────────────────────────────────────────────────
MODEL_ID = "microsoft/Phi-3.5-mini-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading Phi-3.5 Mini...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    attn_implementation="eager",
)

print(f"Loaded — GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

Loaded — GPU: 0.84 GB


In [7]:
# ── Cell 6: LoRA config ──────────────────────────────────────────────────────
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,145,728 || all params: 3,824,225,280 || trainable%: 0.0823


In [ ]:
# ── Cell 7: Train ────────────────────────────────────────────────────────────
wandb.finish()
wandb.init(
    project="scd-medical-llm",
    name="phi35-mini-scd-v2",
    config={
        "model": "Phi-3.5-mini-instruct",
        "dataset": "TeslaInch/SCD-Instruction-Tuning",
        "version": "v2",
        "train_examples": len(train_formatted),
        "val_examples": len(val_formatted),
        "lora_rank": 16,
        "lora_alpha": 32,
        "epochs": 3,
    }
)

sft_config = SFTConfig(
    output_dir="./scd-phi35-adapter-v2",
    num_train_epochs=3,               # reduced from 5 — less forgetting
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    learning_rate=1e-4,               # reduced from 2e-4 — more conservative
    weight_decay=0.05,                # increased from 0.01 — less forgetting
    warmup_steps=20,
    lr_scheduler_type="cosine",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb",
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    processing_class=tokenizer,
)

print("Starting training run 2...")
print(f"Train examples: {len(train_formatted)}")
print(f"Val examples: {len(val_formatted)}")
print(f"Epochs: 3")
print(f"Steps per epoch: {len(train_formatted) // sft_config.per_device_train_batch_size}")

trainer.train()

/tmp/ipykernel_58/2914438011.py:18: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(


Adding EOS to train dataset:   0%|          | 0/353 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/353 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/39 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/39 [00:00<?, ? examples/s]

Starting training run 2...
Train examples: 353
Val examples: 39
Epochs: 3
Steps per epoch: 176


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,1.345328,1.268511,1.248607,0.700367,79979.000000
100,1.105767,1.052153,1.049897,0.738756,160161.000000
